# Interview Question Generator
### Internee.pk — Machine Learning Internship Task

**Objective:** Generate custom technical and behavioral interview questions for
intern candidates, based on job descriptions and an existing question bank,
using a text-generation model (GPT/LLaMA-family).

**Model used:** `google/flan-t5-small` — a small, free, open-source
instruction-tuned text-generation model. It takes a prompt and generates new
text, the same underlying idea as GPT-3/LLaMA, just a much smaller model that
runs on a normal CPU.

**Reliability:** if the model can't load (no internet, first-run download
issue, etc.) the generator automatically falls back to template-based
variation using the existing question bank, so it always produces output.


In [1]:
import random

# ---------------------------------------------------------------------------
# Existing question bank + job descriptions
# (This is the "existing question banks" + "job descriptions" data the task asks for.
#  Replace this with your real bank and real postings.)
# ---------------------------------------------------------------------------

QUESTION_BANK = {
    "python": {
        "technical": [
            "What is the difference between a list and a tuple in Python?",
            "Explain how Python's garbage collection works.",
            "What are decorators, and can you give an example of when you'd use one?",
            "How does Python handle memory management for large data structures?",
        ],
    },
    "sql": {
        "technical": [
            "What is the difference between INNER JOIN and LEFT JOIN?",
            "How would you optimize a slow-running SQL query?",
            "Explain the difference between WHERE and HAVING clauses.",
        ],
    },
    "machine_learning": {
        "technical": [
            "Explain the bias-variance tradeoff in your own words.",
            "How would you handle an imbalanced dataset in a classification problem?",
            "What's the difference between bagging and boosting?",
        ],
    },
    "react": {
        "technical": [
            "What is the difference between state and props in React?",
            "When would you use useEffect versus useMemo?",
            "How does React's virtual DOM improve performance?",
        ],
    },
    "backend": {
        "technical": [
            "What's the difference between REST and GraphQL?",
            "How would you design an API endpoint to handle file uploads safely?",
            "Explain the concept of idempotency in API design.",
        ],
    },
    "general": {
        "behavioral": [
            "Tell me about a time you had to learn a new tool or technology quickly. How did you approach it?",
            "Describe a situation where you disagreed with a teammate's approach. How did you handle it?",
            "Tell me about a time you missed a deadline. What happened, and what did you learn?",
            "Describe a project you're proud of. What was your specific contribution?",
            "How do you prioritize tasks when everything feels urgent?",
        ],
    },
}

JOB_DESCRIPTIONS = {
    "ML Intern": {
        "skills": ["python", "machine_learning"],
        "summary": "Assists in building and evaluating machine learning models, "
                   "cleaning datasets, and running experiments under senior guidance.",
    },
    "Backend Intern": {
        "skills": ["python", "sql", "backend"],
        "summary": "Builds and maintains REST APIs, works with databases, and "
                   "supports backend services for internal tools.",
    },
    "Frontend Intern": {
        "skills": ["react"],
        "summary": "Builds UI components, works with design mockups, and "
                   "integrates frontend with backend APIs.",
    },
}

print("Question bank and job descriptions loaded.")


Question bank and job descriptions loaded.


## 1. Load the Text-Generation Model (with fallback)

We try to load `flan-t5-small`. If it can't load (no internet, etc.), we mark
it unavailable and use template-based generation instead — the system never
crashes or produces nothing.


In [2]:
_generator = None
_MODEL_NAME = "google/flan-t5-small"

def load_model():
    global _generator
    if _generator is not None:
        return _generator
    try:
        from transformers import pipeline
        _generator = pipeline("text2text-generation", model=_MODEL_NAME)
        print(f"Loaded text-generation model: {_MODEL_NAME}")
    except Exception as e:
        print(f"Could not load '{_MODEL_NAME}' ({e}). Falling back to template-based generation.")
        _generator = False
    return _generator

load_model()


/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Could not load 'google/flan-t5-small' (We couldn't connect to 'https://huggingface.co' to load the files, and couldn't find them in the cached files.
Check your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.). Falling back to template-based generation.


False

## 2. Template-Based Fallback

A simple, always-available method: takes a skill name and turns it into a
natural-sounding interview question using sentence templates.


In [3]:
TEMPLATE_STARTERS = [
    "Can you walk me through how you would approach {topic}?",
    "What challenges have you faced when working with {topic}?",
    "How would you explain {topic} to someone new to the field?",
    "Describe a scenario where {topic} would be critical to get right.",
]

def template_fallback_question(skill):
    topic = skill.replace("_", " ")
    template = random.choice(TEMPLATE_STARTERS)
    return template.format(topic=topic)

# quick check
print(template_fallback_question("machine_learning"))


How would you explain machine learning to someone new to the field?


## 3. Real LLM-Based Generation

We prompt the model with the role's job description, the target skill, and a
few real example questions from the bank (few-shot style) — this keeps
generated questions relevant and in a similar tone to your existing bank,
rather than generic or off-topic.


In [4]:
def llm_generate_question(skill, existing_questions, role_summary):
    generator = load_model()
    if not generator:
        return None

    examples = "\n".join(f"- {q}" for q in existing_questions[:3])
    prompt = (
        f"You are writing a technical interview question for a candidate applying "
        f"to this role: {role_summary}\n"
        f"The question should be about: {skill.replace('_', ' ')}.\n"
        f"Here are example questions already used for this skill:\n{examples}\n"
        f"Write one NEW, different technical interview question about {skill.replace('_', ' ')}."
    )

    try:
        result = generator(prompt, max_new_tokens=60, do_sample=True, temperature=0.8)
        text = result[0]["generated_text"].strip()
        return text if text else None
    except Exception as e:
        print(f"Generation failed ({e}), using fallback.")
        return None


## 4. Put It Together: Generate a Full Question Set for a Role


In [5]:
def generate_question_set(role_name, num_technical=3, num_behavioral=2, use_llm=True):
    if role_name not in JOB_DESCRIPTIONS:
        raise ValueError(f"Unknown role '{role_name}'. Available roles: {list(JOB_DESCRIPTIONS.keys())}")

    job = JOB_DESCRIPTIONS[role_name]
    skills = job["skills"]
    summary = job["summary"]

    technical_questions = []
    for i in range(num_technical):
        skill = skills[i % len(skills)]
        bank_questions = QUESTION_BANK.get(skill, {}).get("technical", [])

        question = None
        if use_llm and bank_questions:
            question = llm_generate_question(skill, bank_questions, summary)

        if not question:
            if bank_questions and random.random() < 0.5:
                question = random.choice(bank_questions)
            else:
                question = template_fallback_question(skill)

        technical_questions.append({"skill": skill, "question": question})

    behavioral_pool = QUESTION_BANK["general"]["behavioral"]
    behavioral_questions = random.sample(behavioral_pool, min(num_behavioral, len(behavioral_pool)))

    return {
        "role": role_name,
        "summary": summary,
        "technical_questions": technical_questions,
        "behavioral_questions": behavioral_questions,
    }


## 5. Generate Question Sets for Every Role


In [6]:
for role in JOB_DESCRIPTIONS:
    print(f"\n### {role} ###")
    result = generate_question_set(role, num_technical=3, num_behavioral=2, use_llm=True)
    print(f"Role summary: {result['summary']}\n")

    print("Technical Questions:")
    for i, q in enumerate(result["technical_questions"], 1):
        print(f"  {i}. [{q['skill']}] {q['question']}")

    print("\nBehavioral Questions:")
    for i, q in enumerate(result["behavioral_questions"], 1):
        print(f"  {i}. {q}")



### ML Intern ###
Role summary: Assists in building and evaluating machine learning models, cleaning datasets, and running experiments under senior guidance.

Technical Questions:
  1. [python] What challenges have you faced when working with python?
  2. [machine_learning] What's the difference between bagging and boosting?
  3. [python] What are decorators, and can you give an example of when you'd use one?

Behavioral Questions:
  1. Describe a situation where you disagreed with a teammate's approach. How did you handle it?
  2. Describe a project you're proud of. What was your specific contribution?

### Backend Intern ###
Role summary: Builds and maintains REST APIs, works with databases, and supports backend services for internal tools.

Technical Questions:
  1. [python] How does Python handle memory management for large data structures?
  2. [sql] What is the difference between INNER JOIN and LEFT JOIN?
  3. [backend] What challenges have you faced when working with backend?



## Notes / Next Steps
- Replace `QUESTION_BANK` and `JOB_DESCRIPTIONS` with your real data — the
  more real example questions per skill, the better the LLM's generated
  questions will match your organization's style.
- `flan-t5-small` runs on CPU and downloads automatically the first time
  (~300MB, one-time, needs internet). For higher-quality generation, you can
  swap in a larger model (e.g. `google/flan-t5-base` or `google/flan-t5-large`)
  by changing `_MODEL_NAME` — larger models need more RAM/time.
- For production use with a real GPT-3/GPT-4 quality model, replace
  `llm_generate_question`'s body with an API call to OpenAI or another
  hosted LLM provider instead of a local Hugging Face model.
- The fallback path guarantees the system never fails to return questions,
  even with no internet access or if the model download fails.
